In [1]:
import psycopg2
import pandas as pd
import networkx as nx
import base58
import matplotlib.pyplot as plt
import re
from math import comb
import matplotlib.dates as mdates
import networkx as nx
from collections import defaultdict
from networkx.algorithms.approximation.clique import large_clique_size
from itertools import combinations
import numpy as np
from networkx.algorithms.coloring import greedy_color
import random
from collections import Counter
import requests
import time
from datetime import datetime, timedelta, timezone

In [2]:
conn = psycopg2.connect(
    dbname="sui_indexer",
    user="postgres",
    password="56904628",
    host="172.26.112.1",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()

In [3]:
cur.execute("""
SELECT
    t.checkpoint_sequence AS checkpoint,
    oc.transaction_digest AS tx,
    oc.address AS object_address,
    c.timestamp AS timestamp
FROM transactions t
JOIN object_changes oc
  ON oc.transaction_digest = t.transaction_digest
JOIN checkpoints c
  ON c.sequence_number = t.checkpoint_sequence
WHERE oc.change_type IS NOT NULL;
""")
write_rows = cur.fetchall()

In [ ]:
# graphs[checkpoint] = nx.Graph()
graphs = defaultdict(nx.Graph)

# per checkpoint: object -> list of txs that wrote it
writers_by_cp_obj = defaultdict(lambda: defaultdict(list))

# all txs per checkpoint (so isolated nodes exist)
txs_by_cp = defaultdict(set)

# checkpoint -> timestamp
timestamps_by_cp = {}

for checkpoint, tx, object_address, timestamp in write_rows:
    tx_hex = tx.hex()
    obj_hex = object_address.hex()

    txs_by_cp[checkpoint].add(tx_hex)
    writers_by_cp_obj[checkpoint][obj_hex].append(tx_hex)

    # store timestamp
    if checkpoint not in timestamps_by_cp:
        timestamps_by_cp[checkpoint] = timestamp

# Build graphs
for cp, txs in txs_by_cp.items():
    G = graphs[cp]

    # 1) add all txs as nodes (isolated txs preserved)
    G.add_nodes_from(txs)

    # 2) add edges for shared-write objects
    for writers in writers_by_cp_obj[cp].values():
        writers = list(set(writers))  # de-dup
        if len(writers) < 2:
            continue

        for u, v in combinations(writers, 2):
            G.add_edge(u, v)

# graphs[checkpoint] = NetworkX conflict graph for that checkpoint
print(f"Built {len(graphs)} checkpoint graphs")